In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

# ------------------------------------------------------------
# 1. Tiny nonlinear classification problem
# ------------------------------------------------------------
# y = 1 when x1 and x2 have the same sign
# This is basically an XOR-style problem.

N = 1000
X = torch.randn(N, 2)
y = ((X[:, 0] * X[:, 1]) > 0).long()

X_train, X_val = X[:700], X[700:]
y_train, y_val = y[:700], y[700:]


# ------------------------------------------------------------
# 2. Candidate operations
# ------------------------------------------------------------

class Identity(nn.Module):
    def forward(self, x):
        return x


class Zero(nn.Module):
    def forward(self, x):
        return torch.zeros_like(x)


class LinearOp(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer = nn.Linear(2, 2)

    def forward(self, x):
        return self.layer(x)


class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 8),
            nn.ReLU(),
            nn.Linear(8, 2),
        )

    def forward(self, x):
        return self.net(x)


# ------------------------------------------------------------
# 3. DARTS mixed operation
# ------------------------------------------------------------

class MixedOp(nn.Module):
    def __init__(self):
        super().__init__()

        self.ops = nn.ModuleList([
            Identity(),
            LinearOp(),
            MLP(),
            Zero(),
        ])

        # Architecture parameters α
        # Start equally weighted, like the DARTS paper.
        self.alpha = nn.Parameter(torch.zeros(len(self.ops)))

    def forward(self, x):

        # DARTS continuous relaxation:
        #
        #       exp(alpha_i)
        # p_i = ------------
        #       sum_j exp(alpha_j)
        #
        probs = F.softmax(self.alpha, dim=0)

        result = 0

        for p, op in zip(probs, self.ops):
            result = result + p * op(x)

        return result


# ------------------------------------------------------------
# 4. Tiny network containing the searchable edge
# ------------------------------------------------------------

class TinyDARTS(nn.Module):
    def __init__(self):
        super().__init__()

        self.mixed = MixedOp()
        self.classifier = nn.Linear(2, 2)

    def forward(self, x):
        x = self.mixed(x)
        return self.classifier(x)


model = TinyDARTS()


# ------------------------------------------------------------
# 5. Separate w and α
# ------------------------------------------------------------

alpha_params = [model.mixed.alpha]

weight_params = [
    p for name, p in model.named_parameters()
    if "alpha" not in name
]

weight_optimizer = torch.optim.SGD(
    weight_params,
    lr=0.05
)

alpha_optimizer = torch.optim.Adam(
    alpha_params,
    lr=0.01
)


# ------------------------------------------------------------
# 6. DARTS optimization
# ------------------------------------------------------------

for step in range(501):

    # ========================================================
    # A. Architecture step
    #
    # α <- α - η ∇α L_val(w, α)
    #
    # First-order DARTS approximation:
    # treat current w as the inner optimum.
    # ========================================================

    alpha_optimizer.zero_grad()

    val_logits = model(X_val)
    val_loss = F.cross_entropy(val_logits, y_val)

    val_loss.backward()
    alpha_optimizer.step()


    # ========================================================
    # B. Weight step
    #
    # w <- w - η ∇w L_train(w, α)
    # ========================================================

    weight_optimizer.zero_grad()

    train_logits = model(X_train)
    train_loss = F.cross_entropy(train_logits, y_train)

    train_loss.backward()
    weight_optimizer.step()


    # --------------------------------------------------------
    # Watch the architecture evolve
    # --------------------------------------------------------

    if step % 50 == 0:

        with torch.no_grad():

            probs = F.softmax(model.mixed.alpha, dim=0)

            pred = model(X_val).argmax(dim=1)
            accuracy = (pred == y_val).float().mean()

        print(
            f"\nstep {step:3d}"
            f" | train loss {train_loss.item():.3f}"
            f" | val acc {accuracy.item():.3f}"
        )

        for name, p in zip(
            ["identity", "linear", "MLP", "zero"],
            probs
        ):
            print(f"    {name:8s}: {p.item():.3f}")


# ------------------------------------------------------------
# 7. Discretize architecture
# ------------------------------------------------------------

names = ["identity", "linear", "MLP", "zero"]

probs = F.softmax(model.mixed.alpha, dim=0)

winner = torch.argmax(probs).item()

print("\nFinal architecture:")
print("chosen operation =", names[winner])
print("probabilities    =", probs.detach())